# CFTC USD OIS Swaption Extraction

This notebook extracts and cleans publicly reported USD European SOFR OIS swaption transactions for subsequent Hull–White calibration.

The objective at this stage is only to construct a traceable baseline dataset. Selection by moneyness and calibration suitability is performed later, after each swaption can be valued against the reconstructed SOFR curve.

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

## 1. Raw CFTC swaption data

The source is the CFTC cumulative rates SDR file for 2 September 2026.

USD European OIS swaptions are identified using the corresponding payer (`Call`) and receiver (`Put`) UPI FISNs. Only new interest-rate trades executed on the market date are retained.

The **Execution Timestamp** is used to determine the trade date; the SDR Event Timestamp may correspond to a later reporting or lifecycle event.

In [2]:
market_date = pd.Timestamp("2026-09-02")

raw_path = Path("../data/raw/2026-09-02")
processed_path = Path("../data/processed")

cftc_path = raw_path / "CFTC_CUMULATIVE_RATES_2026_09_02.csv"

cftc = pd.read_csv(cftc_path, low_memory=False)

print("Raw shape:", cftc.shape)

Raw shape: (32064, 110)


In [3]:
date_cols = [
    "Event timestamp",
    "Execution Timestamp",
    "Effective Date",
    "Expiration Date",
    "Maturity date of the underlier",
    "First exercise date",
]

for col in date_cols:
    cftc[col] = pd.to_datetime(
        cftc[col],
        errors="coerce",
        utc=col in ["Event timestamp", "Execution Timestamp"],
    )

In [4]:
# UPI FISN:
#   Call -> payer swaption
#   Put  -> receiver swaption

swaption_fisns = [
    "NA/O Call Epn OIS USD",
    "NA/O P Epn OIS USD",
]

swaption_mask = (
    (cftc["Asset Class"] == "IR")
    & (cftc["Action type"].isin(["NEWT", "NEW"]))
    & (cftc["Event type"] == "TRAD")
    & (cftc["Notional currency-Leg 1"] == "USD")
    & (cftc["UPI FISN"].isin(swaption_fisns))
    & (cftc["Execution Timestamp"].dt.date == market_date.date())
)

swpt = cftc.loc[swaption_mask].copy()

print("Raw same-day USD OIS swaptions:", len(swpt))
print(swpt["UPI FISN"].value_counts())

Raw same-day USD OIS swaptions: 212
UPI FISN
NA/O P Epn OIS USD       107
NA/O Call Epn OIS USD    105
Name: count, dtype: int64


## 2. Contract fields and derived quantities

Premium, notional $\mathcal{N}$ and strike $K$ fields are converted to numeric form. Option expiry $T_{\mathrm{exp}}$ is measured from the market date to the reported First exercise date; underlying maturity $T_n$ is measured from the market date to the underlier maturity. Both use ACT/365F.

The project assumes the underlying swap starts at expiry, $T_0=T_{\mathrm{exp}}$, so its tenor is $T_n-T_0$. This is a zero start-date lag; matching option exercise and expiration dates does not independently establish the swap's effective date.

`Call` FISNs are classified as payer swaptions and `Put` FISNs as receiver swaptions. The premium per unit notional is

$$v^{\mathrm{mkt}}=\frac{\mathrm{Premium}}{\mathcal{N}}.$$

It is retained as a diagnostic here and used for pricing comparisons in Notebook 03.

In [5]:
def parse_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

swpt["premium"] = parse_numeric(swpt["Option Premium Amount"])

swpt["notional"] = parse_numeric(swpt["Notional amount-Leg 1"])

swpt["strike"] = parse_numeric(swpt["Strike Price"])

In [ ]:
# T_exp: option expiry measured from valuation time 0.
swpt["option_expiry_years"] = (swpt["First exercise date"] - market_date).dt.days / 365.0

# T_n - T_0, assuming T_0 = T_exp (zero start-date lag).
swpt["underlying_tenor_years"] = (
    swpt["Maturity date of the underlier"] - swpt["First exercise date"]
).dt.days / 365.0

# Call = payer swaption, Put = receiver swaption.
swpt["payer"] = swpt["UPI FISN"] == "NA/O Call Epn OIS USD"

# v_mkt: reported premium divided by notional.
swpt["premium_per_notional"] = swpt["premium"] / swpt["notional"]

In [ ]:
print("Missing first exercise date:", swpt["First exercise date"].isna().sum())

print("Missing expiration date:", swpt["Expiration Date"].isna().sum())

print("Missing underlying maturity:", swpt["Maturity date of the underlier"].isna().sum())

valid_dates = (
    swpt["First exercise date"].notna() & swpt["Maturity date of the underlier"].notna()
)

print(
    "Underlying maturity <= expiry:",
    (
        valid_dates
        & (swpt["Maturity date of the underlier"] <= swpt["First exercise date"])
    ).sum(),
)

Missing first exercise date: 0
Missing expiration date: 0
Missing underlying maturity: 41
Underlying maturity <= expiry: 0


In [ ]:
european_date_check = swpt[
    swpt["First exercise date"].notna() & swpt["Expiration Date"].notna()
]

print(
    "First exercise != expiration:",
    (
        european_date_check["First exercise date"] != european_date_check["Expiration Date"]
    ).sum(),
)

First exercise != expiration: 0


## 3. Baseline cleaning

The baseline sample excludes observations with:

- non-standard contract terms;
- package transactions;
- missing or non-positive premiums;
- missing or non-positive notionals;
- missing strikes;
- expired options;
- missing or invalid underlying maturities.

No ATM or moneyness filter is applied here. Moneyness depends on the forward swap rate implied by the reconstructed SOFR curve and is assessed in the subsequent calibration-basket notebook.

In [ ]:
# Baseline exclusions:
#   - explicitly non-standard contracts
#   - package transactions
#   - zero / missing premiums
#   - zero / missing notionals
#   - missing strikes
#   - expired options
#   - invalid underlying maturities

clean = swpt.copy()

print("Start:", len(clean))

clean = clean[clean["Non-standardized term indicator"] == False].copy()
print("Standard terms:", len(clean))

clean = clean[clean["Package indicator"] == False].copy()
print("Non-package:", len(clean))

clean = clean[clean["premium"].notna() & (clean["premium"] > 0)].copy()
print("Positive premium:", len(clean))

clean = clean[clean["notional"].notna() & (clean["notional"] > 0)].copy()
print("Valid notional:", len(clean))

clean = clean[clean["strike"].notna()].copy()
print("Valid strike:", len(clean))

clean = clean[clean["First exercise date"] > market_date].copy()
print("Future expiry:", len(clean))

clean = clean[clean["Maturity date of the underlier"] > clean["First exercise date"]].copy()
print("Valid underlying maturity:", len(clean))

Start: 212
Standard terms: 187
Non-package: 73
Positive premium: 54
Valid notional: 45
Valid strike: 45
Future expiry: 45
Valid underlying maturity: 28


## 4. Clean sample diagnostics

The remaining observations are inspected for:

- expiry and underlying-tenor coverage;
- payer/receiver balance;
- strike, premium and notional ranges;
- potential large-notional reporting flags.

The large-notional flag is retained for later review rather than used as an automatic exclusion criterion at this stage.

In [10]:
summary_cols = [
    "UPI FISN",
    "option_expiry_years",
    "underlying_tenor_years",
    "strike",
    "premium",
    "notional",
    "premium_per_notional",
    "Large notional off-facility swap election indicator",
]

clean[summary_cols].describe(include="all")

,UPI FISN,option_expiry_years,underlying_tenor_years,strike,premium,notional,premium_per_notional,Large notional off-facility swap election indicator
count,28,28.000000,28.000000,28.000000,2.800000e+01,2.800000e+01,28.000000,26
unique,2,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NA/O Call Epn OIS USD,NaN,NaN,NaN,NaN,NaN,NaN,False
freq,14,NaN,NaN,NaN,NaN,NaN,NaN,24
mean,NaN,1.423386,12.196869,0.044885,1.299078e+06,1.169643e+08,0.020557,NaN
std,NaN,1.694680,8.767004,0.002923,1.068984e+06,1.114874e+08,0.019505,NaN
min,NaN,0.063014,1.008219,0.038000,9.000000e+04,2.000000e+07,0.000325,NaN
25%,NaN,0.063014,7.016438,0.043354,6.045000e+05,5.000000e+07,0.004687,NaN
50%,NaN,1.000000,10.016438,0.044353,8.305000e+05,9.750000e+07,0.009050,NaN
75%,NaN,2.260274,12.521233,0.046200,1.959375e+06,1.300000e+08,0.037366,NaN


In [11]:
# Payer / receiver balance

clean["UPI FISN"].value_counts()

UPI FISN
NA/O Call Epn OIS USD    14
NA/O P Epn OIS USD       14
Name: count, dtype: int64

In [ ]:
# Clean expiry x tenor coverage

clean["expiry_bucket"] = clean["option_expiry_years"].round(1)
clean["tenor_bucket"] = clean["underlying_tenor_years"].round(1)

clean_grid = pd.crosstab(clean["expiry_bucket"], clean["tenor_bucket"])

clean_grid

tenor_bucket,1.0,2.0,5.0,7.0,10.0,20.0,30.0
expiry_bucket,,,,,,,
0.1,0,0,1,6,4,1,1
1.0,0,0,0,0,3,0,2
2.0,0,0,0,0,0,2,1
3.0,0,1,0,0,2,0,0
4.0,0,0,0,0,1,0,0
4.9,2,0,0,0,0,0,0
5.0,0,0,0,0,1,0,0


## 5. Final extraction dataset

Only fields required for instrument identification, pricing, calibration and auditability are retained.

The dissemination identifier and original reported contract fields are preserved so that individual observations can be traced back to the source data if required.

In [13]:
keep_cols = [
    # Traceability / instrument identity
    "Dissemination Identifier",
    "Execution Timestamp",
    "Unique Product Identifier",
    "UPI FISN",
    "UPI Underlier Name",

    # Contract dates
    "First exercise date",
    "Expiration Date",
    "Maturity date of the underlier",

    # Economics
    "Strike Price",
    "Option Premium Amount",
    "Option Premium Currency",
    "Notional amount-Leg 1",
    "Notional currency-Leg 1",

    # Useful cleaned/derived fields
    "strike",
    "premium",
    "notional",
    "payer",
    "option_expiry_years",
    "underlying_tenor_years",
    "premium_per_notional",

    # Keep for audit / possible later exclusion
    "Large notional off-facility swap election indicator",
]

clean_swaptions = (
    clean[keep_cols]
    .sort_values(["option_expiry_years", "underlying_tenor_years"])
    .reset_index(drop=True)
)

## 6. Export

The cleaned swaption dataset is exported for the next stage of the project.

The next step is to combine these observations with the frozen 2 September 2026 SOFR curve, calculate each underlying forward swap rate, measure swaption moneyness, and select a defensible Hull–White calibration basket.

In [14]:
swaption_path = processed_path / "cftc_clean_swaptions_2026-09-02.csv"

clean_swaptions.to_csv(swaption_path, index=False)

print(f"Saved swaptions to: {swaption_path}")

Saved swaptions to: ..\data\processed\cftc_clean_swaptions_2026-09-02.csv
